# DTW K-Means 클러스터링 + 섹터 단위 특성 분석 (Colab 버전)

**목표**
- 종목별 수익률 시계열을 DTW(Dynamic Time Warping) 거리 기반으로 클러스터링
- 업종과 무관하게 실제 가격 움직임 패턴이 유사한 종목 그룹 발견
- 클러스터 결과 → FGC(Fourier Graph Convolution) 그래프 엣지 설계에 활용

**입력**: `data/kospi_valid.parquet`  
**출력**: `data/cluster_assignments.csv`, `data/cluster_centroids.npy`

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q tslearn scikit-learn pyarrow

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from tslearn.clustering import TimeSeriesKMeans
from tslearn.utils import to_time_series_dataset

warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

DRIVE_ROOT  = Path('/content/drive/MyDrive/grad_project')
DATA_DIR    = DRIVE_ROOT / 'data'
parquet_path = DATA_DIR / 'kospi_valid.parquet'

# 클러스터링 설정
CLUSTER_START = '2022-01-01'
CLUSTER_END   = '2024-12-31'
MIN_DAYS      = 700          # 해당 기간 최소 영업일
K_RANGE       = range(5, 21) # 최적 K 탐색 범위
FINAL_K       = 12           # 최종 클러스터 수 (실루엣 분석 후 조정)
RANDOM_STATE  = 42

assert parquet_path.exists(), f'파일 없음: {parquet_path}'
print(f'✓ {parquet_path}  ({parquet_path.stat().st_size/1e6:.1f} MB)')

## 1. 데이터 로드 & 클러스터링용 시계열 준비

In [ ]:
df = pd.read_parquet(parquet_path)
df['Date'] = pd.to_datetime(df['Date'])

# 클러스터링 기간 필터
df_period = df[
    (df['Date'] >= CLUSTER_START) &
    (df['Date'] <= CLUSTER_END)
].copy()

# 해당 기간 데이터 충분한 종목만
valid_codes = (
    df_period.groupby('종목코드').size()
    .loc[lambda s: s >= MIN_DAYS].index
)
df_period = df_period[df_period['종목코드'].isin(valid_codes)]

print(f'클러스터링 대상: {len(valid_codes)}개 종목')
print(f'기간: {CLUSTER_START} ~ {CLUSTER_END}')

# 종목별 일간 수익률 피벗 (날짜 × 종목)
ret_pivot = df_period.pivot_table(
    index='Date', columns='종목코드', values='Return_1d'
).sort_index()

# 공통 날짜만 유지 + 결측 보간
ret_pivot = ret_pivot.fillna(0)
print(f'수익률 행렬: {ret_pivot.shape}  (날짜 × 종목)')

In [ ]:
# 종목별 누적 수익률로 변환 (패턴 비교에 더 적합)
cum_ret = (1 + ret_pivot).cumprod() - 1  # (T_daily, N)

# ── 월별 다운샘플링 (일별 ~700포인트 → 월말 ~35포인트) ─────────────────
# DTW 거리 계산이 O(T²)이므로 T를 줄이면 속도가 ~400배 빨라짐
# 3년치 월별 누적수익률로도 macro 패턴(상승/하락 구간, 변동성 클러스터) 충분히 구분 가능
cum_ret_monthly = cum_ret.resample('ME').last()   # (T_monthly, N)
print(f'다운샘플링: {cum_ret.shape[0]}일 → {cum_ret_monthly.shape[0]}개월')

# 각 종목 z-score 정규화 (스케일 차이 제거)
series_matrix = cum_ret_monthly.values.T          # (N, T_monthly)
scaler = StandardScaler()
series_norm = scaler.fit_transform(series_matrix.T).T  # (N, T_monthly)

codes_list = list(ret_pivot.columns)
T          = series_norm.shape[1]
print(f'입력 시계열: {series_norm.shape}  (종목 × 월)')

# tslearn 포맷으로 변환 (N, T_monthly, 1)
X_ts = to_time_series_dataset(series_norm)

## 2. 최적 K 탐색 (실루엣 분석)

In [ ]:
%%time
# 실루엣 점수로 최적 K 선택
# 시간 절약: Sakoe-Chiba band(w=10) + euclidean metric으로 빠르게 탐색
sil_scores = {}
inertias   = {}

for k in K_RANGE:
    km = TimeSeriesKMeans(
        n_clusters=k,
        metric='softdtw',       # soft-DTW: GPU 활용 가능, 미분 가능
        metric_params={'gamma': 0.1},
        max_iter=10,            # 탐색용 빠른 반복
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    labels = km.fit_predict(X_ts)
    # 실루엣: DTW 대신 euclidean으로 근사 계산 (속도)
    sil = silhouette_score(series_norm, labels, metric='euclidean')
    sil_scores[k] = sil
    inertias[k]   = km.inertia_
    print(f'  K={k:2d} | 실루엣: {sil:.4f} | 관성: {km.inertia_:.2f}')

best_k = max(sil_scores, key=sil_scores.get)
print(f'\n최고 실루엣 K = {best_k}  (score: {sil_scores[best_k]:.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ks = list(sil_scores.keys())
axes[0].plot(ks, [sil_scores[k] for k in ks], 'o-', color='steelblue', lw=2)
axes[0].axvline(best_k, color='red', linestyle='--', label=f'최적 K={best_k}')
axes[0].set_xlabel('클러스터 수 (K)')
axes[0].set_ylabel('실루엣 점수')
axes[0].set_title('실루엣 점수로 최적 K 선택')
axes[0].legend()

axes[1].plot(ks, [inertias[k] for k in ks], 's-', color='tomato', lw=2)
axes[1].axvline(best_k, color='red', linestyle='--', label=f'최적 K={best_k}')
axes[1].set_xlabel('클러스터 수 (K)')
axes[1].set_ylabel('관성 (Inertia)')
axes[1].set_title('Elbow Method')
axes[1].legend()

plt.tight_layout()
plt.show()

FINAL_K = best_k
print(f'FINAL_K = {FINAL_K} 으로 설정')

## 3. 최종 클러스터링 (Soft-DTW)

In [ ]:
%%time
km_final = TimeSeriesKMeans(
    n_clusters=FINAL_K,
    metric='softdtw',
    metric_params={'gamma': 0.1},
    max_iter=50,
    tol=1e-4,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=True,
)
labels = km_final.fit_predict(X_ts)
centroids = km_final.cluster_centers_  # (K, T, 1)

print(f'\n클러스터별 종목 수:')
unique, counts = np.unique(labels, return_counts=True)
for k, c in zip(unique, counts):
    print(f'  클러스터 {k:2d}: {c}종목')

## 4. 클러스터 결과 시각화

In [ ]:
# 클러스터 중심 + 샘플 시계열 시각화
n_cols = 4
n_rows = (FINAL_K + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3), sharex=True)
axes = axes.flat

# 월별 x축 레이블 (분기 단위)
month_labels = cum_ret_monthly.index
tick_pos   = list(range(0, T, 3))
tick_labels = [month_labels[i].strftime('%y.%m') for i in tick_pos]
colors = plt.cm.tab20(np.linspace(0, 1, FINAL_K))

for k in range(FINAL_K):
    ax   = axes[k]
    mask = labels == k
    members = series_norm[mask]  # (n_k, T)

    # 클러스터 내 샘플 (최대 10개) 연하게
    for s in members[:10]:
        ax.plot(s, color=colors[k], alpha=0.15, lw=0.8)

    # 클러스터 중심
    centroid = centroids[k, :, 0]
    ax.plot(centroid, color=colors[k], lw=2.5)
    ax.axhline(0, color='gray', lw=0.5, linestyle='--')
    ax.set_xticks(tick_pos)
    ax.set_xticklabels(tick_labels, fontsize=7, rotation=30)
    ax.set_title(f'Cluster {k}  (n={mask.sum()})', fontsize=9)

# 빈 subplot 제거
for ax in list(axes)[FINAL_K:]:
    ax.set_visible(False)

plt.suptitle(f'DTW K-Means 클러스터 중심 (K={FINAL_K}, 2022~2024, 월별)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. 섹터 × 클러스터 구성 분석

In [ ]:
# 클러스터 배정 DataFrame
sector_map = (
    df[['종목코드', '종목명', '업종']]
    .drop_duplicates('종목코드')
    .set_index('종목코드')
)

cluster_df = pd.DataFrame({
    '종목코드': codes_list,
    'cluster':  labels,
}).merge(sector_map.reset_index(), on='종목코드', how='left')

print(cluster_df['cluster'].value_counts().sort_index())

In [ ]:
# 클러스터 × 업종 히트맵
top_sectors = cluster_df['업종'].value_counts().head(15).index
ct = pd.crosstab(
    cluster_df['cluster'],
    cluster_df['업종']
)[top_sectors]

# 비율로 정규화 (클러스터 내 업종 비중)
ct_pct = ct.div(ct.sum(axis=1), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.heatmap(
    ct, ax=axes[0], cmap='Blues', annot=True, fmt='d',
    annot_kws={'size': 8},
    xticklabels=[s[:10] for s in top_sectors]
)
axes[0].set_title('클러스터 × 업종 (종목 수)')
axes[0].set_xlabel('업종')
axes[0].set_ylabel('클러스터')
axes[0].tick_params(axis='x', rotation=30)

sns.heatmap(
    ct_pct, ax=axes[1], cmap='YlOrRd', annot=True, fmt='.2f',
    annot_kws={'size': 8},
    xticklabels=[s[:10] for s in top_sectors]
)
axes[1].set_title('클러스터 × 업종 (비율)')
axes[1].set_xlabel('업종')
axes[1].set_ylabel('클러스터')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# 클러스터별 수익률 통계
cluster_stats = cluster_df.merge(
    df.groupby('종목코드').agg(
        ann_return=('Return_1d', lambda x: x.mean() * 252),
        ann_vol=('Return_1d', lambda x: x.std() * np.sqrt(252)),
    ).reset_index(),
    on='종목코드'
)

stats_summary = cluster_stats.groupby('cluster').agg(
    종목수=('종목코드', 'count'),
    연간수익률=('ann_return', 'mean'),
    연간변동성=('ann_vol', 'mean'),
    샤프=('ann_return', lambda x: x.mean() / cluster_stats.loc[x.index, 'ann_vol'].mean()),
).round(4)

print(stats_summary.to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = plt.cm.tab20(np.linspace(0, 1, FINAL_K))

for ax, col, title in [
    (axes[0], '연간수익률', '클러스터별 평균 연간 수익률'),
    (axes[1], '연간변동성', '클러스터별 평균 연간 변동성'),
    (axes[2], '샤프',       '클러스터별 샤프 지수'),
]:
    vals = stats_summary[col]
    bar_colors = [colors[i] for i in range(FINAL_K)]
    ax.bar(range(FINAL_K), vals, color=bar_colors, alpha=0.85, edgecolor='k', lw=0.5)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xticks(range(FINAL_K))
    ax.set_xlabel('클러스터')
    ax.set_title(title, fontsize=10)
    if col != '샤프':
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))

plt.tight_layout()
plt.show()

## 6. FGC용 그래프 엣지 행렬 생성

In [ ]:
# 같은 클러스터 내 종목끼리 연결 → 인접 행렬
N = len(codes_list)
adj_matrix = np.zeros((N, N), dtype=np.float32)

for k in range(FINAL_K):
    idx = np.where(labels == k)[0]
    for i in idx:
        for j in idx:
            if i != j:
                adj_matrix[i, j] = 1.0

# 수익률 상관관계로 엣지 가중치 부여 (선택)
corr_matrix = np.corrcoef(series_norm)  # (N, N)
weighted_adj = adj_matrix * np.clip(corr_matrix, 0, None)  # 양의 상관만

print(f'인접 행렬: {adj_matrix.shape}')
print(f'평균 연결 수 (클러스터 내): {adj_matrix.sum(axis=1).mean():.1f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(adj_matrix, cmap='Blues', aspect='auto')
axes[0].set_title('클러스터 기반 인접 행렬')
axes[0].set_xlabel('종목 인덱스')
axes[0].set_ylabel('종목 인덱스')

axes[1].imshow(weighted_adj, cmap='RdBu_r', aspect='auto', vmin=0, vmax=1)
axes[1].set_title('상관관계 가중 인접 행렬')
axes[1].set_xlabel('종목 인덱스')
plt.colorbar(plt.cm.ScalarMappable(cmap='RdBu_r'), ax=axes[1])

plt.tight_layout()
plt.show()

## 7. 결과 저장

In [ ]:
# 클러스터 배정 저장
cluster_df.to_csv(DATA_DIR / 'cluster_assignments.csv', index=False, encoding='utf-8-sig')

# 클러스터 중심 저장
np.save(DATA_DIR / 'cluster_centroids.npy', centroids)

# 인접 행렬 저장 (FGC 학습에 사용)
np.save(DATA_DIR / 'adj_matrix.npy',      adj_matrix)
np.save(DATA_DIR / 'weighted_adj.npy',    weighted_adj)

# 종목 코드 순서 저장 (인접 행렬과 매핑)
pd.Series(codes_list, name='종목코드').to_csv(
    DATA_DIR / 'cluster_code_order.csv', index=True, header=True
)

print('저장 완료:')
print(f'  cluster_assignments.csv  → 종목별 클러스터 배정')
print(f'  cluster_centroids.npy   → 클러스터 중심 시계열 ({centroids.shape})')
print(f'  adj_matrix.npy          → FGC 인접 행렬 ({adj_matrix.shape})')
print(f'  weighted_adj.npy        → 상관관계 가중 인접 행렬')
print(f'  cluster_code_order.csv  → 행렬 인덱스-종목코드 매핑')
print()
print('▶ 다음 단계 (8월): 시계열 백본 비교 실험 (Chronos / PatchTST / StockMixer)')